In [17]:
from dotenv import load_dotenv
import os
from supabase import create_client, Client
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
import football_analytics.data_processing.preprocessing as preprocessing
from importlib import reload
import json
reload(preprocessing)

<module 'football_analytics.data_processing.preprocessing' from 'C:\\Users\\david\\Documents\\Git\\football-analytics\\src\\football_analytics\\data_processing\\preprocessing.py'>

## Loading Supabase Info

In [2]:
# Load variables from .env into the environment
load_dotenv()

# Read variables
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_KEY")

In [3]:
# Initialize client
supabase: Client = create_client(supabase_url, supabase_key)

## Process Statsbomb Data

#### Shots

In [15]:
table_name = "shots"
response = supabase.table(table_name).select("*").limit(1).execute()
print(response)

data=[{'created_at': '2025-12-08T20:57:48.743902+00:00', 'statsbomb_event_id': '91387214-dcee-44c8-b43c-046e2fecca89', 'x1': 110.4, 'y1': 27.6, 'distance_to_goal': 15.681836627130123, 'angle_to_goal_deg': 18.470825950050802, 'keeper_distance': 5.954829972383757, 'min_defender_distance': 3.3541019662496834, 'avg_defender_distance': 12.337133253225018, 'num_def_in_shot_triangle': 0, 'num_teammates_in_box': 0, 'shot_to_min_def_ratio': 4.6615223867938544, 'shot_type': 'Open Play', 'body_part': 'Left Foot', 'outcome': 'Saved', 'full_json': '{"id":"91387214-dcee-44c8-b43c-046e2fecca89","index":1409,"period":1,"timestamp":1766363710181,"minute":35,"second":10,"type":{"id":16,"name":"Shot"},"possession":61,"possession_team":{"id":779,"name":"Argentina"},"play_pattern":{"id":1,"name":"Regular Play"},"team":{"id":779,"name":"Argentina"},"duration":0.321314,"tactics":null,"related_events":["23ebb06b-af2b-451c-a742-951c2af31db1"],"player":{"id":29560,"name":"Juli\\u00e1n \\u00c1lvarez"},"position"

In [18]:
folder_path = Path("../../data/statsbomb/open-data-master/data/events")
json_files = list(folder_path.glob("*.json"))

table_name = "shots"
BATCHSIZE_SHOT_UPSERT = 1000  # tune to 500–2000 depending on timeouts
MATCH_LIMIT = 1e6
skip_first_n_matches = -1

shots_to_upsert = []

for match_index, json_file in tqdm(enumerate(json_files), desc="Processing...", total=len(json_files)):
    if match_index < skip_first_n_matches:
        continue
    if match_index > MATCH_LIMIT:
        break

    match_id = json_file.stem

    # Load events without pandas
    events = json.loads(json_file.read_bytes())
    shot_events = [e for e in events if e.get("shot")]

    # Extract features
    for event in shot_events:
        shot_data = preprocessing.extract_shot_features(event, match_id)
        shots_to_upsert.append(shot_data)

    # Upsert in larger batches to reduce overhead
    if len(shots_to_upsert) >= BATCHSIZE_SHOT_UPSERT:
        # returning="minimal" avoids row payloads (if your client supports it)
        supabase.table(table_name).upsert(
            shots_to_upsert,
            on_conflict="statsbomb_event_id",
            returning="minimal",
        ).execute()
        shots_to_upsert = []

# Flush remainder
if shots_to_upsert:
    supabase.table(table_name).upsert(
        shots_to_upsert,
        on_conflict="statsbomb_event_id",
        returning="minimal",
    ).execute()
    shots_to_upsert = []

Processing...:   0%|          | 0/3464 [00:00<?, ?it/s]

#### Competion

In [19]:
json_file = "../../data/statsbomb/open-data-master/data/competitions.json"
with open(json_file, 'r') as f:
    competitions_to_upsert = json.load(f)

In [20]:
table_name = "competitions"
response = supabase.table(table_name).upsert(competitions_to_upsert, on_conflict="competition_id, season_id").execute()

#### Teams

In [22]:
table_name = "teams"
matches_folder = Path("../../data/statsbomb/open-data-master/data/matches")
match_files = list(matches_folder.glob("*/*.json"))
teams_by_id = {}

for match_file in tqdm(match_files, desc="Collecting teams", total=len(match_files)):
    matches = json.loads(match_file.read_bytes())
    for match in matches:
        for side in ("home_team", "away_team"):
            team_row = preprocessing.extract_team_row(match.get(side))
            if team_row and team_row.get("team_id"):
                teams_by_id[team_row["team_id"]] = team_row

teams_df = pd.DataFrame(teams_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
teams_df = teams_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
teams_df = teams_df.where(pd.notnull(teams_df), None)
teams_to_upsert = []
for row in teams_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    teams_to_upsert.append(clean_row)



In [27]:
response = supabase.table(table_name).upsert(teams_to_upsert, on_conflict="team_id").execute()
print(f"Prepared {len(teams_to_upsert)} teams for upsert")


Prepared 312 teams for upsert


#### Players

In [ ]:
table_name = "players"
import math
events_folder = Path("../../data/statsbomb/open-data-master/data/events")
event_files = list(events_folder.glob("*.json"))
players_by_id = {}

for event_file in tqdm(event_files, desc="Collecting players", total=len(event_files)):
    events = json.loads(event_file.read_bytes())
    for ev in events:
        player_row = preprocessing.extract_player_row(ev)
        if player_row and player_row.get("statsbomb_player_id"):
            players_by_id[player_row["statsbomb_player_id"]] = player_row

players_df = pd.DataFrame(players_by_id.values())
# Replace NaN/NaT/inf with None to keep JSON serializable
players_df = players_df.replace([pd.NA, pd.NaT, float("inf"), float("-inf")], None)
players_df = players_df.where(pd.notnull(players_df), None)



In [17]:
players_to_upsert = []
for row in players_df.to_dict(orient="records"):
    clean_row = {}
    for k, v in row.items():
        if pd.isna(v):
            clean_row[k] = None
            continue
        if isinstance(v, (float, int)) and math.isinf(v):
            clean_row[k] = None
            continue
        clean_row[k] = v
    players_to_upsert.append(clean_row)



In [19]:
if players_to_upsert:
    response = supabase.table(table_name).upsert(players_to_upsert, on_conflict="statsbomb_player_id").execute()
else:
    print("No players collected")
print(f"Prepared {len(players_to_upsert)} players for upsert")


Prepared 9043 players for upsert


#### Matches